# Get the new small molecules from ChEMBL36
This shows whether there is some generalization

In [ ]:
import chembl_downloader
import sqlite3
import pandas as pd

path36 = chembl_downloader.download_extract_sqlite(version="36")
path35 = chembl_downloader.download_extract_sqlite(version="35")
conn36 = sqlite3.connect(path36)
conn35 = sqlite3.connect(path35)

query = """
SELECT md.chembl_id, cs.standard_inchi_key, cs.canonical_smiles
FROM molecule_dictionary md
JOIN compound_structures cs
  ON md.molregno = cs.molregno;
"""
df36 = pd.read_sql(query, conn36)
df35 = pd.read_sql(query, conn35)
new36 = df36.loc[
    ~df36['standard_inchi_key'].isin(df35['standard_inchi_key'])
]
new36.to_csv("data/chembl36_new_full_raw.csv",index=False)

In [ ]:
import chembl_downloader
import sqlite3
import pandas as pd

path36 = chembl_downloader.download_extract_sqlite(version="36")
path35 = chembl_downloader.download_extract_sqlite(version="35")
conn36 = sqlite3.connect(path36)
conn35 = sqlite3.connect(path35)

query = """
SELECT
    md.chembl_id,
    md.molregno,
    cs.standard_inchi_key,
    cs.canonical_smiles
FROM molecule_dictionary md
JOIN compound_structures cs
    ON md.molregno = cs.molregno
WHERE
    cs.standard_inchi_key IS NOT NULL
    AND NOT EXISTS (
        SELECT 1
        FROM compound_structural_alerts csa
        WHERE csa.molregno = md.molregno
    );
"""
df36 = pd.read_sql(query, conn36)
df35 = pd.read_sql(query, conn35)
new36f = df36.loc[
    ~df36['standard_inchi_key'].isin(df35['standard_inchi_key'])
]
new36f.to_csv("data/chembl36_new_filtered_raw.csv",index=False)

In [ ]:
import chembl_downloader
from rdkit import Chem
from rdkit.Chem.MolStandardize import rdMolStandardize
from rdkit.Chem import Descriptors
import csv
from tqdm.auto import tqdm
from rdkit import RDLogger

RDLogger.DisableLog('rdApp.*')


def standardize(smiles):
    """
    see:
    https://github.com/greglandrum/RSC_OpenScience_Standardization_202104/tree/main
    and
    https://github.com/PatWalters/practical_cheminformatics_tutorials/blob/main/misc/working_with_ChEMBL_drug_data.ipynb
    see this paper for the chembl workflow and the justification to not touch tautomers
    https://jcheminf.biomedcentral.com/articles/10.1186/s13321-020-00456-1
    """
    try:
        mol = Chem.MolFromSmiles(smiles)
        clean_mol = rdMolStandardize.Cleanup(mol) 
        parent_clean_mol = rdMolStandardize.FragmentParent(clean_mol)
        if Descriptors.ExactMolWt(parent_clean_mol) < 1000:
            uncharger = rdMolStandardize.Uncharger()
            uncharged_parent_clean_mol = uncharger.uncharge(parent_clean_mol)
            #include tautomer enumeration if needed
            #te = rdMolStandardize.TautomerEnumerator()
            #uncharged_parent_clean_mol = te.Canonicalize(uncharged_parent_clean_mol)
            smi = Chem.MolToSmiles(uncharged_parent_clean_mol)
        else:
            smi = None
    except Exception as e:
        smi = None
        print("warning some error for",smiles)
        print("the error was",e)
    return smi
tqdm.pandas(desc="mb")
new36f["canonical_smiles"] = new36f["canonical_smiles"].progress_apply(standardize)
unique_smis = list(set(new36f["canonical_smiles"]))
with open('data/chembl36_new_filtered.csv', 'w', newline='') as csvfile:
    writer = csv.writer(csvfile,delimiter=',',quotechar='"', quoting=csv.QUOTE_MINIMAL)
    for smi in unique_smis:
        try:
            len(smi)
            if smi:
                writer.writerow([smi])
        except Exception as e:
            print("XXX",e)

In [ ]:
unique_smis[136616]

In [ ]:
import chembl_downloader
from rdkit import Chem
from rdkit.Chem.MolStandardize import rdMolStandardize
from rdkit.Chem import Descriptors
import csv
from tqdm.auto import tqdm
from rdkit import RDLogger

RDLogger.DisableLog('rdApp.*')


def standardize(smiles):
    """
    see:
    https://github.com/greglandrum/RSC_OpenScience_Standardization_202104/tree/main
    and
    https://github.com/PatWalters/practical_cheminformatics_tutorials/blob/main/misc/working_with_ChEMBL_drug_data.ipynb
    see this paper for the chembl workflow and the justification to not touch tautomers
    https://jcheminf.biomedcentral.com/articles/10.1186/s13321-020-00456-1
    """
    try:
        mol = Chem.MolFromSmiles(smiles)
        clean_mol = rdMolStandardize.Cleanup(mol) 
        parent_clean_mol = rdMolStandardize.FragmentParent(clean_mol)
        if Descriptors.ExactMolWt(parent_clean_mol) < 1000:
            uncharger = rdMolStandardize.Uncharger()
            uncharged_parent_clean_mol = uncharger.uncharge(parent_clean_mol)
            #include tautomer enumeration if needed
            #te = rdMolStandardize.TautomerEnumerator()
            #uncharged_parent_clean_mol = te.Canonicalize(uncharged_parent_clean_mol)
            smi = Chem.MolToSmiles(uncharged_parent_clean_mol)
        else:
            smi = None
    except Exception as e:
        smi = None
        print("warning some error for",smiles)
        print("the error was",e)
    return smi
tqdm.pandas(desc="mb")
new36["canonical_smiles"] = new36["canonical_smiles"].progress_apply(standardize)
unique_smis = list(set(new36["canonical_smiles"]))
with open('data/chembl36_new_full.csv', 'w', newline='') as csvfile:
    writer = csv.writer(csvfile,delimiter=',',quotechar='"', quoting=csv.QUOTE_MINIMAL)
    for smi in unique_smis:
        try:
            len(smi)
            if smi:
                writer.writerow([smi])
        except Exception as e:
            print("XXX",e)